# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/News Video Classification"

VIDEO_DIR = os.path.join(BASE_PATH, "Downloaded_Videos")

videos = sorted([
    f for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(".mp4")
])

print("Videos found:", len(videos))
print("First 10:", videos[:10])
print("Last 10:", videos[-10:])

## Imports

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project Configuration

In [ ]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

import os

# ----------------------------
# Base Path
# ----------------------------

BASE_PATH = "/content/drive/MyDrive/News Video Classification"


# ----------------------------
# Dataset
# ----------------------------

DATASET_PATH = os.path.join(
    BASE_PATH,
    "NewsVideoDataset.xlsx"
)


# ----------------------------
# Folder Structure
# ----------------------------

# Final preprocessed video clips
VIDEO_DIR = os.path.join(
    BASE_PATH,
    "Downloaded_Videos"
)

# Extracted representative keyframes
KEYFRAMES_DIR = os.path.join(
    BASE_PATH,
    "Keyframes"
)

FEATURES_DIR = os.path.join(
    BASE_PATH,
    "Features"
)

DINO_DIR = os.path.join(
    FEATURES_DIR,
    "DINOv2"
)

MOBILE_DIR = os.path.join(
    FEATURES_DIR,
    "MobileNetV3"
)

# Pipeline outputs
OUTPUTS_DIR = os.path.join(
    BASE_PATH,
    "Outputs"
)

# Per-video segmentation/keyframe metadata
JSON_DIR = os.path.join(
    OUTPUTS_DIR,
    "JSON"
)

# Optional segmentation graphs
GRAPHS_DIR = os.path.join(
    OUTPUTS_DIR,
    "Graphs"
)


# ----------------------------
# Shot Segmentation Parameters
# ----------------------------

NO_OF_BINS = 16
FRAME_SKIP = 28
THRESHOLD = 0.5


# ----------------------------
# Display Options
# ----------------------------

# Keep False during batch processing
SHOW_GRAPHS = False

SHOW_KEYFRAMES = False


# ----------------------------
# Create folders if missing
# ----------------------------

folders = [
    KEYFRAMES_DIR,

    FEATURES_DIR,
    DINO_DIR,
    MOBILE_DIR,

    OUTPUTS_DIR,
    JSON_DIR,
    GRAPHS_DIR
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)


# ----------------------------
# Verification
# ----------------------------

videos = sorted([
    f for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(".mp4")
])

print("Project configuration loaded successfully!")
print("Video directory:", VIDEO_DIR)
print("Videos found:", len(videos))

# Load Dataset

In [ ]:
import pandas as pd

df = pd.read_excel(DATASET_PATH)

print(f"Total Videos: {len(df)}")

df.head()

# Import VideoSegmentation

In [ ]:
import importlib.util
import os

module_path = os.path.join(BASE_PATH, "video_segmentor.py")

spec = importlib.util.spec_from_file_location(
    "video_segmentor",
    module_path
)

video_segmentor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(video_segmentor)

VideoSegmentation = video_segmentor.VideoSegmentation

print("VideoSegmentation imported successfully!")

#Helper Functions

In [ ]:
def get_video_duration(video_path):
    """
    Returns the duration of a video in seconds.

    Parameters
    ----------
    video_path : str
        Path to the video file.

    Returns
    -------
    float
        Duration of the video in seconds.
        Returns 0 if the video cannot be opened.
    """

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    cap.release()

    if fps == 0:
        return 0

    duration = frame_count / fps

    return duration

In [ ]:
def run_segmentation(video_id, clip_path):
    """
    Performs shot boundary detection on a video clip.

    Parameters
    ----------
    video_id : str
        Video ID from the dataset.

    clip_path : str
        Path to the extracted clip.

    Returns
    -------
    dict or None
        Segmentation results or None if failed.
    """

    try:

        segmentor = VideoSegmentation(
            path=clip_path,
            no_of_bins=NO_OF_BINS,
            frame_skip=FRAME_SKIP,
            threshold=THRESHOLD
        )

        (
            distances,
            fps,
            frame_skip,
            duration,
            segment_frames,
            segment_times
        ) = segmentor.segment_video("Bhattachrya Distance")

        # Show graph only if enabled
        if SHOW_GRAPHS:

            segmentor.plotting(
                distances,
                fps,
                frame_skip,
                "Bhattachrya Distance"
            )

        return {
            "distances": distances,
            "fps": fps,
            "frame_skip": frame_skip,
            "duration": duration,
            "segment_frames": segment_frames,
            "segment_times": segment_times
        }

    except Exception as e:

        print(f"[ERROR] Segmentation failed for {video_id}: {e}")

        return None

In [ ]:
def get_keyframe_indices(segment_frames, total_frames):
    """
    Computes representative keyframe indices using the middle frame
    of each detected shot.

    Parameters
    ----------
    segment_frames : list
        Frame numbers where new shots begin.

    total_frames : int
        Total number of frames in the video.

    Returns
    -------
    list
        List of representative keyframe indices.
    """

    boundaries = [0] + segment_frames + [total_frames - 1]

    keyframe_indices = []

    for i in range(len(boundaries) - 1):

        start = boundaries[i]
        end = boundaries[i + 1]

        middle = (start + end) // 2

        keyframe_indices.append(middle)

    return keyframe_indices

In [ ]:
def save_keyframes(
    video_id,
    clip_path,
    keyframe_indices,
    output_folder=None
):
    """
    Saves representative keyframes from a video clip.

    Parameters
    ----------
    video_id : str
        Video ID from the dataset.

    clip_path : str
        Path to the extracted video clip.

    keyframe_indices : list
        List of frame indices to save.

    Returns
    -------
    list
        Information about each saved keyframe.
    """

    if output_folder is None:

        output_folder = os.path.join(
            KEYFRAMES_DIR,
            video_id
        )

    os.makedirs(
        output_folder,
        exist_ok=True
    )

    cap = cv2.VideoCapture(clip_path)

    saved_keyframes = []

    for shot_number, frame_index in enumerate(keyframe_indices, start=1):

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

        success, frame = cap.read()

        if success:

            image_name = f"shot_{shot_number}.jpg"
            image_path = os.path.join(output_folder, image_name)

            cv2.imwrite(image_path, frame)

            saved_keyframes.append({
                "shot": shot_number,
                "frame": frame_index,
                "image": image_name
            })

    cap.release()

    return saved_keyframes

In [ ]:
def save_json(video_id, segmentation_results, keyframes):
    """
    Saves preprocessing results as a JSON file.

    Parameters
    ----------
    video_id : str
        Video ID from the dataset.

    segmentation_results : dict
        Output returned by run_segmentation().

    keyframes : list
        Output returned by save_keyframes().

    Returns
    -------
    str
        Path to the saved JSON file.
    """

    json_path = os.path.join(
        JSON_DIR,
        f"{video_id}.json"
    )

    data = {
        "video_id": video_id,
        "duration": segmentation_results["duration"],
        "fps": segmentation_results["fps"],
        "frame_skip": segmentation_results["frame_skip"],
        "num_shots": len(keyframes),
        "segment_frames": segmentation_results["segment_frames"],
        "segment_times": segmentation_results["segment_times"],
        "keyframes": keyframes
    }

    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

    return json_path

In [ ]:
def cleanup_files(original_video=None, clip_video=None):
    """
    Deletes temporary video files based on project settings.

    Parameters
    ----------
    original_video : str, optional
        Path to the downloaded original video.

    clip_video : str, optional
        Path to the extracted clip.
    """

    if DELETE_ORIGINAL_VIDEO:

        if original_video and os.path.exists(original_video):
            os.remove(original_video)

    # Uncomment later if you also want to remove clips
    # if DELETE_CLIP_VIDEO:
    #     if clip_video and os.path.exists(clip_video):
    #         os.remove(clip_video)

#Core Pipeline Functions


In [ ]:
def process_single_video(video_id):
    """
    Runs the visual preprocessing pipeline for one already-downloaded clip.

    Pipeline:
        Downloaded_Videos/<video_id>.mp4
            -> shot segmentation
            -> middle-frame keyframe selection
            -> save keyframes
            -> save JSON metadata

    Parameters
    ----------
    video_id : str
        Video ID from the dataset.

    Returns
    -------
    dict
        Processing summary.
    """

    print(f"\nProcessing {video_id}...")

    # ------------------------------------------------
    # Step 1: Locate downloaded clip
    # ------------------------------------------------

    clip_video = os.path.join(
        VIDEO_DIR,
        f"{video_id}.mp4"
    )

    if not os.path.exists(clip_video):
        print(f"[SKIP] {video_id} - video file not found")

        return {
            "video_id": video_id,
            "status": "Skipped",
            "reason": "Video file not found"
        }

    # ------------------------------------------------
    # Step 2: Shot Segmentation
    # ------------------------------------------------

    segmentation = run_segmentation(
        video_id,
        clip_video
    )

    if segmentation is None:
        return {
            "video_id": video_id,
            "status": "Failed",
            "reason": "Segmentation failed"
        }

    # ------------------------------------------------
    # Step 3: Get total number of frames
    # ------------------------------------------------

    cap = cv2.VideoCapture(clip_video)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    cap.release()

    if total_frames <= 0:
        return {
            "video_id": video_id,
            "status": "Failed",
            "reason": "Could not read video frames"
        }

    # ------------------------------------------------
    # Step 4: Compute middle-frame keyframes
    # ------------------------------------------------

    keyframe_indices = get_keyframe_indices(
        segmentation["segment_frames"],
        total_frames
    )

    # ------------------------------------------------
    # Step 5: Save Keyframes
    # ------------------------------------------------

    keyframes = save_keyframes(
        video_id,
        clip_video,
        keyframe_indices
    )

    # ------------------------------------------------
    # Step 6: Save JSON metadata
    # ------------------------------------------------

    json_path = save_json(
        video_id,
        segmentation,
        keyframes
    )

    print(
        f"[SUCCESS] {video_id} | "
        f"Shots: {len(keyframes)} | "
        f"Keyframes: {len(keyframes)}"
    )

    return {
        "video_id": video_id,
        "status": "Success",
        "num_shots": len(keyframes),
        "num_keyframes": len(keyframes),
        "json": json_path
    }

#Batch Processing

In [ ]:
results = []

for _, row in df.iterrows():

    result = process_single_video(
        row["Video_ID"]
    )

    results.append(result)

#Summary

In [ ]:
summary_df = pd.DataFrame(results)

display(summary_df)

summary_path = os.path.join(
    OUTPUTS_DIR,
    "processing_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print(f"\nSummary saved to:\n{summary_path}")

In [ ]:
print("STATUS COUNTS")
print(summary_df["status"].value_counts())

print("\nTOTAL KEYFRAMES")
print(summary_df["num_keyframes"].sum())

print("\nTOTAL SHOTS")
print(summary_df["num_shots"].sum())

print("\nFAILED / SKIPPED VIDEOS")
display(
    summary_df[
        summary_df["status"] != "Success"
    ]
)

# Backbone Comparison
## DINOv2-Small vs MobileNetV3-Small

This section compares DINOv2-Small with a lightweight MobileNetV3-Small
backbone for visual feature extraction from representative video keyframes.

Both models are evaluated using the same input resolution and keyframe set.
The comparison considers computational complexity, parameter count,
feature dimensionality, and inference time.

In [ ]:
import torch
import torchvision
import time
import numpy as np

from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:", device)

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%capture

!pip -q install fvcore

In [ ]:
import torch

dinov2 = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

dinov2 = dinov2.to(device)
dinov2.eval()

print("DINOv2-Small loaded successfully!")

In [ ]:
from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

mobilenet_weights = MobileNet_V3_Small_Weights.DEFAULT

mobilenet = mobilenet_v3_small(
    weights=mobilenet_weights
)

# We want features, not ImageNet class predictions
mobilenet.classifier = torch.nn.Identity()

mobilenet = mobilenet.to(device)
mobilenet.eval()

print("MobileNetV3-Small loaded successfully!")

In [ ]:
print(
    "DINOv2 device:",
    next(dinov2.parameters()).device
)

print(
    "MobileNet device:",
    next(mobilenet.parameters()).device
)

In [ ]:
from fvcore.nn import FlopCountAnalysis

# Same dummy input for both models
dummy_input = torch.randn(
    1, 3, 224, 224
).to(device)


def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


def profile_model(model, input_tensor):

    # FLOPs
    flops = FlopCountAnalysis(
        model,
        input_tensor
    )

    total_flops = flops.total()

    # Convert FLOPs -> GFLOPs
    gflops = total_flops / 1e9

    # Parameters
    params = count_parameters(model)
    params_million = params / 1e6

    return params_million, gflops

In [ ]:
dinov2_params, dinov2_gflops = profile_model(
    dinov2,
    dummy_input
)

mobilenet_params, mobilenet_gflops = profile_model(
    mobilenet,
    dummy_input
)

print("DINOv2-Small")
print(f"Parameters: {dinov2_params:.2f} M")
print(f"GFLOPs: {dinov2_gflops:.2f}")

print("\nMobileNetV3-Small")
print(f"Parameters: {mobilenet_params:.2f} M")
print(f"GFLOPs: {mobilenet_gflops:.2f}")

In [ ]:
with torch.no_grad():

    dinov2_features = dinov2(
        dummy_input
    )

    mobilenet_features = mobilenet(
        dummy_input
    )

print(
    "DINOv2 output shape:",
    dinov2_features.shape
)

print(
    "MobileNetV3 output shape:",
    mobilenet_features.shape
)

In [ ]:
print(
    "DINOv2 feature dimension:",
    dinov2_features.shape[-1]
)

print(
    "MobileNetV3 feature dimension:",
    mobilenet_features.shape[-1]
)

In [ ]:
import time
import torch

def benchmark(model, input_tensor, warmup=20, runs=100):
    model.eval()

    with torch.no_grad():

        # Warm-up
        for _ in range(warmup):
            _ = model(input_tensor)

        torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(runs):
            _ = model(input_tensor)

        torch.cuda.synchronize()

        end = time.perf_counter()

    avg_ms = ((end - start) / runs) * 1000

    return avg_ms

In [ ]:
dinov2_time = benchmark(dinov2, dummy_input)
mobilenet_time = benchmark(mobilenet, dummy_input)

print(f"DINOv2-Small: {dinov2_time:.2f} ms/image")
print(f"MobileNetV3-Small: {mobilenet_time:.2f} ms/image")

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Backbone": [
        "DINOv2-Small",
        "MobileNetV3-Small"
    ],
    "Parameters (M)": [
        round(dinov2_params, 2),
        round(mobilenet_params, 2)
    ],
    "GFLOPs (fvcore)": [
        round(dinov2_gflops, 2),
        round(mobilenet_gflops, 2)
    ],
    "Feature Dimension": [
        dinov2_features.shape[-1],
        mobilenet_features.shape[-1]
    ],
    "Inference Time (ms/image)": [
        round(dinov2_time, 2),
        round(mobilenet_time, 2)
    ]
})

display(comparison_df)

# Visual Feature Extraction
This section extracts visual features from the representative keyframes
using both DINOv2-Small and MobileNetV3-Small.

For each video:

- Load all representative keyframes.
- Extract feature vectors.
- Aggregate them using mean pooling.
- Save one feature vector per video.

In [ ]:
from PIL import Image
from torchvision import transforms
import numpy as np
import os

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
def extract_feature(model, image_path):

    image = Image.open(image_path).convert("RGB")

    tensor = transform(image)

    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():

        feature = model(tensor)

    feature = feature.squeeze()

    feature = feature.cpu().numpy()

    return feature

In [ ]:
def aggregate_video_features(
        model,
        video_folder):

    images = sorted([
        f for f in os.listdir(video_folder)
        if f.endswith(".jpg")
    ])

    features = []

    for img in images:

        img_path = os.path.join(
            video_folder,
            img
        )

        feat = extract_feature(
            model,
            img_path
        )

        features.append(feat)

    features = np.stack(features)

    video_feature = np.mean(
        features,
        axis=0
    )

    return video_feature

In [ ]:
# ============================================================
# EXTRACT FEATURES FOR ALL VIDEOS
# ============================================================

import os
import numpy as np

# Create output folders
os.makedirs(DINO_DIR, exist_ok=True)
os.makedirs(MOBILE_DIR, exist_ok=True)

processed = []

video_folders = sorted([
    f for f in os.listdir(KEYFRAMES_DIR)
    if os.path.isdir(os.path.join(KEYFRAMES_DIR, f))
])

print(f"Found {len(video_folders)} videos.\n")

for video_id in video_folders:

    print(f"Processing {video_id}...")

    folder = os.path.join(
        KEYFRAMES_DIR,
        video_id
    )

    # -----------------------------
    # DINOv2
    # -----------------------------

    dino_feature = aggregate_video_features(
        dinov2,
        folder
    )

    np.save(
        os.path.join(
            DINO_DIR,
            f"{video_id}.npy"
        ),
        dino_feature
    )

    # -----------------------------
    # MobileNetV3
    # -----------------------------

    mobile_feature = aggregate_video_features(
        mobilenet,
        folder
    )

    np.save(
        os.path.join(
            MOBILE_DIR,
            f"{video_id}.npy"
        ),
        mobile_feature
    )

    processed.append(video_id)

print("\nFeature extraction completed!")
print(f"Videos processed: {len(processed)}")

# Build Master Training Dataset

This section combines the extracted visual features with the
annotated labels from the dataset to create the final training
datasets for both DINOv2 and MobileNetV3.

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
dataset = pd.read_excel(DATASET_PATH)

print(dataset.shape)

dataset.head()

In [ ]:
available_videos = sorted([
    f.replace(".npy", "")
    for f in os.listdir(DINO_DIR)
])

print(len(available_videos))

In [ ]:
dataset = dataset[
    dataset["Video_ID"].isin(
        available_videos
    )
].reset_index(drop=True)

print(dataset.shape)

In [ ]:
import numpy as np
import pandas as pd
import os

def build_master_dataset(feature_dir, feature_dim):
    rows = []

    for _, row in dataset.iterrows():

        video_id = row["Video_ID"]

        feature_path = os.path.join(
            feature_dir,
            f"{video_id}.npy"
        )

        if not os.path.exists(feature_path):
            continue

        feature = np.load(feature_path)

        sample = {
            "Video_ID": video_id
        }

        # Feature columns
        for i in range(feature_dim):
            sample[f"f{i}"] = feature[i]

        # Labels
        sample["Content_Format"] = row["Content_Format"]
        sample["Source_Type"] = row["Source_Type"]
        sample["Primary_Language"] = row["Primary_Language"]
        sample["Geography"] = row["Geography"]
        sample["Production"] = row["Production"]

        rows.append(sample)

    return pd.DataFrame(rows)

In [ ]:
dino_master = build_master_dataset(
    DINO_DIR,
    384
)

mobile_master = build_master_dataset(
    MOBILE_DIR,
    576
)

print("DINO:", dino_master.shape)
print("MobileNet:", mobile_master.shape)

display(dino_master.head())

In [ ]:
MASTER_DIR = os.path.join(BASE_PATH, "Master_Datasets")
os.makedirs(MASTER_DIR, exist_ok=True)

dino_master.to_csv(
    os.path.join(MASTER_DIR, "DINO_Master.csv"),
    index=False
)

mobile_master.to_csv(
    os.path.join(MASTER_DIR, "MobileNet_Master.csv"),
    index=False
)

print("Master datasets saved successfully!")

# Train Classifier

In [ ]:
dino_df = pd.read_csv(
    os.path.join(
        MASTER_DIR,
        "DINO_Master.csv"
    )
)

mobile_df = pd.read_csv(
    os.path.join(
        MASTER_DIR,
        "MobileNet_Master.csv"
    )
)

print(dino_df.shape)
print(mobile_df.shape)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
%%capture

!pip -q install joblib

In [ ]:
def train_and_evaluate(dataset, target, backbone_name):

    print("=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)

    # -------------------------
    # Features
    # -------------------------

    feature_columns = [
        c for c in dataset.columns
        if c.startswith("f")
    ]

    X = dataset[feature_columns].values

    # -------------------------
    # Labels
    # -------------------------

    encoder = LabelEncoder()

    y = encoder.fit_transform(
        dataset[target]
    )

    class_names = encoder.classes_

    # -------------------------
    # Cross Validation
    # -------------------------

    skf = StratifiedKFold(
        n_splits=2,
        shuffle=True,
        random_state=42
    )

    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []

    final_cm = None

    fold = 1

    for train_idx, test_idx in skf.split(X, y):

        print(f"\nFold {fold}")

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # -------------------------
        # Classifier
        # -------------------------

        clf = RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )

        clf.fit(
            X_train,
            y_train
        )

        predictions = clf.predict(
            X_test
        )

        # -------------------------
        # Metrics
        # -------------------------

        acc = accuracy_score(
            y_test,
            predictions
        )

        prec = precision_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        rec = recall_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)

        final_cm = confusion_matrix(
            y_test,
            predictions
        )

        fold += 1

    # -----------------------------------
    # SAVE MODEL
    # -----------------------------------

    import joblib

    model_folder = (
        DINO_MODEL_DIR
        if backbone_name == "DINOv2"
        else MOBILE_MODEL_DIR
    )

    joblib.dump(
        clf,
        os.path.join(
            model_folder,
            f"{target}_model.pkl"
        )
    )

    joblib.dump(
        encoder,
        os.path.join(
            model_folder,
            f"{target}_encoder.pkl"
        )
    )

    # -----------------------------------
    # RESULTS
    # -----------------------------------

    results = {

        "Backbone": backbone_name,

        "Target": target,

        "Accuracy": np.mean(accuracies),

        "Precision": np.mean(precisions),

        "Recall": np.mean(recalls),

        "F1": np.mean(f1_scores),

        "Confusion_Matrix": final_cm,

        "Classes": class_names

    }

    return results

In [ ]:
# ============================================================
# MODEL DIRECTORIES
# ============================================================

import os

MODEL_DIR = os.path.join(BASE_PATH, "Models")

DINO_MODEL_DIR = os.path.join(
    MODEL_DIR,
    "DINOv2"
)

MOBILE_MODEL_DIR = os.path.join(
    MODEL_DIR,
    "MobileNetV3"
)

os.makedirs(DINO_MODEL_DIR, exist_ok=True)
os.makedirs(MOBILE_MODEL_DIR, exist_ok=True)

print("Model folders created successfully!")

# Train All Models

In [ ]:
backbones = {
    "DINOv2": dino_df,
    "MobileNetV3": mobile_df
}

targets = [
    "Production",
    "Source_Type",
    "Primary_Language",
    "Geography",
    "Content_Format"
]

all_results = []

for backbone_name, dataset in backbones.items():

    print("\n" + "=" * 70)
    print(backbone_name)
    print("=" * 70)

    for target in targets:

        result = train_and_evaluate(
            dataset,
            target,
            backbone_name
        )

        all_results.append(result)

In [ ]:
results_df = pd.DataFrame([
    {
        "Backbone": r["Backbone"],
        "Target": r["Target"],
        "Accuracy": round(r["Accuracy"], 4),
        "Precision": round(r["Precision"], 4),
        "Recall": round(r["Recall"], 4),
        "F1": round(r["F1"], 4)
    }
    for r in all_results
])

display(results_df)

In [ ]:
results_df.to_csv(
    os.path.join(
        OUTPUTS_DIR,
        "Backbone_Comparison.csv"
    ),
    index=False
)

print("Comparison table saved.")

# Final Model Training

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
TARGETS = [
    "Production",
    "Source_Type",
    "Primary_Language",
    "Geography",
    "Content_Format"
]

In [ ]:
for target in TARGETS:

    print("=" * 60)
    print(target)
    print("=" * 60)

    feature_cols = [
        c for c in dino_df.columns
        if c.startswith("f")
    ]

    X = dino_df[feature_cols]

    encoder = LabelEncoder()

    y = encoder.fit_transform(
        dino_df[target]
    )

    clf = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    clf.fit(X, y)

    joblib.dump(
        clf,
        os.path.join(
            DINO_MODEL_DIR,
            f"{target}_model.pkl"
        )
    )

    joblib.dump(
        encoder,
        os.path.join(
            DINO_MODEL_DIR,
            f"{target}_encoder.pkl"
        )
    )

    print("Saved.")

# Final Inference Pipeline

This section demonstrates metadata prediction for an unseen news video using the trained DINOv2-based visual classifier.

Pipeline:

Unknown Video
→ Shot Segmentation
→ Representative Keyframe Extraction
→ DINOv2 Feature Extraction
→ Metadata Prediction

## Unknown Video Inference

In [ ]:
# ============================================================
# INFERENCE CONFIGURATION
# ============================================================

INFERENCE_DIR = os.path.join(
    BASE_PATH,
    "Inference"
)

INFERENCE_VIDEO_DIR = os.path.join(
    INFERENCE_DIR,
    "Videos"
)

INFERENCE_KEYFRAME_DIR = os.path.join(
    INFERENCE_DIR,
    "Keyframes"
)

os.makedirs(INFERENCE_VIDEO_DIR, exist_ok=True)
os.makedirs(INFERENCE_KEYFRAME_DIR, exist_ok=True)

print("Inference folders created successfully!")

In [ ]:
# ============================================================
# LOAD UNKNOWN VIDEO
# ============================================================

videos = sorted([
    f for f in os.listdir(INFERENCE_VIDEO_DIR)
    if f.lower().endswith(".mp4")
])

print("Available Videos:\n")

for i, video in enumerate(videos, start=1):
    print(f"{i}. {video}")

choice = int(input("\nSelect video number: "))

video_name = videos[choice - 1]

unknown_video = os.path.join(
    INFERENCE_VIDEO_DIR,
    video_name
)

print("\nSelected:", video_name)     # Change to unknown2.mp4 whenever you want

unknown_video = os.path.join(
    INFERENCE_VIDEO_DIR,
    video_name
)

print("Testing video:", video_name)

if not os.path.exists(unknown_video):
    raise FileNotFoundError(
        f"Video not found:\n{unknown_video}"
    )

print("Unknown video loaded successfully!")
print(unknown_video)

In [ ]:
# ============================================================
# CLEAN PREVIOUS KEYFRAMES
# ============================================================

import shutil

if os.path.exists(INFERENCE_KEYFRAME_DIR):

    for item in os.listdir(INFERENCE_KEYFRAME_DIR):

        path = os.path.join(
            INFERENCE_KEYFRAME_DIR,
            item
        )

        if os.path.isfile(path):
            os.remove(path)

        elif os.path.isdir(path):
            shutil.rmtree(path)

print("Previous keyframes removed.")

In [ ]:
# ============================================================
# SHOT SEGMENTATION
# ============================================================

segmentation = run_segmentation(
    "UNKNOWN",
    unknown_video
)

if segmentation is None:
    raise RuntimeError("Segmentation failed!")

print("Segmentation completed successfully!")
print("Number of shots:", len(segmentation["segment_frames"]))
print("Video duration:", segmentation["duration"])

In [ ]:
# ============================================================
# COMPUTE KEYFRAME INDICES
# ============================================================

cap = cv2.VideoCapture(unknown_video)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

cap.release()

keyframe_indices = get_keyframe_indices(
    segmentation["segment_frames"],
    total_frames
)

print("Total frames:", total_frames)
print("Representative keyframes:", len(keyframe_indices))
print(keyframe_indices)

In [ ]:
# ============================================================
# SAVE REPRESENTATIVE KEYFRAMES
# ============================================================

saved_keyframes = save_keyframes(
    video_id="UNKNOWN",
    clip_path=unknown_video,
    keyframe_indices=keyframe_indices,
    output_folder=INFERENCE_KEYFRAME_DIR
)

print("Keyframes saved:", len(saved_keyframes))

In [ ]:
# ============================================================
# EXTRACT DINO FEATURES
# ============================================================

unknown_folder = INFERENCE_KEYFRAME_DIR

dino_feature = aggregate_video_features(
    dinov2,
    unknown_folder
)

print(dino_feature.shape)

In [ ]:
import joblib
import os

targets = [
    "Production",
    "Source_Type",
    "Primary_Language",
    "Geography",
    "Content_Format"
]

models = {}
encoders = {}

for target in targets:

    models[target] = joblib.load(
        os.path.join(
            DINO_MODEL_DIR,
            f"{target}_model.pkl"
        )
    )

    encoders[target] = joblib.load(
        os.path.join(
            DINO_MODEL_DIR,
            f"{target}_encoder.pkl"
        )
    )

print("All models loaded successfully!")

In [ ]:
X = dino_feature.reshape(1, -1)

predictions = {}

for target in targets:

    pred = models[target].predict(X)

    label = encoders[target].inverse_transform(pred)

    predictions[target] = label[0]

In [ ]:
print("=" * 60)
print("PREDICTED METADATA")
print("=" * 60)

for key, value in predictions.items():
    print(f"{key:20}: {value}")

In [ ]:
from IPython.display import display
from PIL import Image

print("Representative Keyframes\n")

for image_name in sorted(os.listdir(INFERENCE_KEYFRAME_DIR)):

    image_path = os.path.join(
        INFERENCE_KEYFRAME_DIR,
        image_name
    )

    print(image_name)

    display(Image.open(image_path))